# BMIN 5200 — Week 6 in-class exercise
## A forward and backward chaining engine in 40 lines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LINK::github-repo/blob/main/exercises/week06.ipynb)

**Time:** ~25 minutes · **Pairs with:** Rules and knowledge-based systems (production systems, chaining, conflict resolution)

### What you'll do
- Represent a rule base as a list of plain Python dicts, exactly the way the "Rule Basics" slide draws it
- Write `forward_chain` and `backward_chain` and count how many facts each one derives to answer the same question
- Watch two rules match the same patient and reach opposite conclusions, then fix it with a conflict resolution strategy
- Build a `why()` facility that reconstructs the derivation of any fact the engine concluded

### Why it matters
Next week you use CLIPS, which is a production system with thirty years of engineering in it. Everything CLIPS does, you are about to do by hand in about forty lines, so that when the agenda and the conflict set show up in clipspy they are recognizable rather than magical. The forward-versus-backward contrast is not academic either: a sepsis surveillance rule engine running forward over every patient in the hospital and a diagnostic consultation system chaining backward from one question are the same rules run in opposite directions, and they cost wildly different amounts.

## Setup

Nothing to install and nothing to import beyond one pretty-printer. The entire inference engine in
this notebook is plain Python built out of dicts, sets, and lists.

In [ ]:
# Nothing to install. A rule is a dict, a fact is a string, a fact base is a set of strings,
# and the whole inference engine is plain Python.
from pprint import pprint

## Part 1 — The rule base and forward chaining

A rule here is a dict with three keys: `name`, `antecedents` (a list of facts that must all be
true), and `consequent` (the fact you may then add). That is the "Rule Base" slide with Python
syntax. Our knowledge base is a fever and sepsis workup: fourteen rules that lead toward a sepsis
decision, plus ten that cover other things the same patient triggers — kidney injury, fall
precautions, nutrition. That mixture is deliberate and it is what a real rule base looks like.

In [ ]:
def rule(name, antecedents, consequent):
    return {"name": name, "antecedents": antecedents, "consequent": consequent}


SEPSIS_RULES = [
    # The path toward a sepsis decision.
    rule("fever",             ["temp_above_38_3"],                                "fever"),
    rule("tachycardia",       ["heart_rate_above_90"],                            "tachycardia"),
    rule("tachypnea",         ["resp_rate_above_20"],                             "tachypnea"),
    rule("leukocytosis",      ["wbc_above_12"],                                   "leukocytosis"),
    rule("sirs-fever-hr",     ["fever", "tachycardia"],                           "sirs_criteria_met"),
    rule("sirs-rr-wbc",       ["tachypnea", "leukocytosis"],                      "sirs_criteria_met"),
    rule("infection-urine",   ["urine_culture_positive"],                         "infection_suspected"),
    rule("infection-chest",   ["chest_xray_infiltrate", "productive_cough"],      "infection_suspected"),
    rule("sepsis",            ["sirs_criteria_met", "infection_suspected"],       "sepsis"),
    rule("hypoperfusion",     ["lactate_above_4"],                                "tissue_hypoperfusion"),
    rule("hypotension",       ["map_below_65"],                                   "hypotension"),
    rule("septic-shock",      ["sepsis", "hypotension", "tissue_hypoperfusion"],  "septic_shock"),
    rule("start-antibiotics", ["sepsis"],                                         "give_broad_spectrum_antibiotics"),
    rule("icu-transfer",      ["septic_shock"],                                   "transfer_to_icu"),

    # Everything else the same admitted patient happens to trigger.
    rule("aki",               ["creatinine_rise_0_3"],                            "acute_kidney_injury"),
    rule("nephrotoxin-hold",  ["acute_kidney_injury"],                            "hold_nephrotoxins"),
    rule("contrast-caution",  ["acute_kidney_injury"],                            "flag_contrast_caution"),
    rule("dvt-prophylaxis",   ["immobile_over_48h"],                              "dvt_prophylaxis_indicated"),
    rule("fall-risk",         ["age_above_75", "immobile_over_48h"],              "fall_precautions"),
    rule("delirium-screen",   ["age_above_75"],                                   "delirium_screening_due"),
    rule("nutrition-consult", ["albumin_below_3", "los_above_5_days"],            "nutrition_consult"),
    rule("pressure-ulcer",    ["immobile_over_48h", "albumin_below_3"],           "pressure_ulcer_risk"),
    rule("pharmacy-review",   ["home_meds_above_8"],                              "pharmacy_reconciliation"),
    rule("social-work",       ["lives_alone", "age_above_75"],                    "social_work_referral"),
]

# One synthetic patient, expressed as the fact base the engine starts from.
PATIENT_FACTS = {
    "temp_above_38_3", "heart_rate_above_90", "resp_rate_above_20", "wbc_above_12",
    "urine_culture_positive", "lactate_above_4", "map_below_65",
    "creatinine_rise_0_3", "immobile_over_48h", "age_above_75",
    "albumin_below_3", "los_above_5_days", "home_meds_above_8", "lives_alone",
}

print(f"{len(SEPSIS_RULES)} rules in the rule base")
print(f"{len(PATIENT_FACTS)} facts known about the patient at admission")
print()
print("One rule, in full:")
pprint(SEPSIS_RULES[8])

Forward chaining is the "Forward Chaining Walkthrough" slide as a loop: sweep the rule base, fire
every rule whose antecedents are all satisfied, and repeat until a whole sweep adds nothing new.
Two details matter. We skip a rule whose consequent is already known, which is what stops the
engine looping forever on `fever -> fever`. And we record, for each derived fact, which rule
produced it — that record is the inference chain, and Part 4 turns it into an explanation.

In [ ]:
def forward_chain(rules, facts):
    """Fire until a full sweep adds nothing. Returns (all facts, inference chain, rules tested)."""
    derived = set(facts)
    chain = {}                      # derived fact -> the rule that produced it
    rules_tested = 0

    something_fired = True
    while something_fired:
        something_fired = False
        for r in rules:
            rules_tested += 1
            if r["consequent"] in derived:
                continue            # Already known; refiring would never terminate.
            if all(antecedent in derived for antecedent in r["antecedents"]):
                derived.add(r["consequent"])
                chain[r["consequent"]] = r
                something_fired = True
    return derived, chain, rules_tested


all_facts, forward_inference_chain, forward_tests = forward_chain(SEPSIS_RULES, PATIENT_FACTS)
newly_derived = sorted(all_facts - PATIENT_FACTS)

print(f"started with {len(PATIENT_FACTS)} facts, finished with {len(all_facts)}")
print(f"derived {len(newly_derived)} new facts in {forward_tests} rule tests")
print()
for fact in newly_derived:
    print(f"  {fact:<35} from rule '{forward_inference_chain[fact]['name']}'")

The engine reached `transfer_to_icu`, which is the answer we actually wanted. It also concluded
that this patient needs pressure ulcer precautions, a pharmacy reconciliation, and a social work
referral. Nothing is wrong with any of those conclusions — they are all correct. They are simply
not what anyone asked. Forward chaining is data-driven: it does not know what your question is,
so it derives everything derivable.

## Part 2 — Backward chaining, and why both strategies exist

Backward chaining starts from the question. To prove a goal, either it is already a known fact,
or some rule concludes it and every antecedent of that rule can itself be proved — which is the
recursive structure on the "Backward Chaining Walkthrough" slide. Two guards keep it honest: a
`stack` of goals currently being pursued, so a rule base containing a cycle cannot recurse
forever, and a count of goals visited so we can compare the cost against forward chaining.

The placeholder below is not recursive. It only reports whether the goal happens to be a starting
fact, so it will fail on anything that needs a rule at all. Make it recursive.

In [ ]:
def backward_chain(rules, facts, goal, chain=None, stack=None, stats=None):
    """Try to prove `goal`. Returns (proved, inference chain, stats)."""
    if chain is None:
        chain = {}
    if stack is None:
        stack = set()               # goals currently being pursued, to break cycles
    if stats is None:
        stats = {"goals_visited": 0}

    stats["goals_visited"] += 1

    if goal in facts:
        return True, chain, stats
    if goal in stack:
        return False, chain, stats  # already trying to prove this; a cycle, not a proof

    # TODO: replace the line below with the recursive case.
    #   For every rule whose consequent is this goal, try to prove all of its antecedents
    #   (passing `stack | {goal}` down so cycles are caught). If they all succeed, record
    #   chain[goal] = that rule and return True. If no rule works, return False.
    return False, chain, stats


proved, backward_inference_chain, backward_stats = backward_chain(
    SEPSIS_RULES, PATIENT_FACTS, "transfer_to_icu")

print(f"goal: transfer_to_icu")
print(f"proved: {proved}")
print(f"goals visited: {backward_stats['goals_visited']}")
print(f"rules used in the proof: {len(backward_inference_chain)}")
if not proved:
    print()
    print("False after one goal is what the placeholder does: transfer_to_icu is not a starting")
    print("fact, and nothing recursive has been written yet. Fill in the TODO and rerun.")

### Predict before you run

Once your `backward_chain` works, we will ask both engines the same single question: *should this
patient go to the ICU?* Commit to numbers before running the next cell.

1. Forward chaining derived 22 new facts to answer it. How many facts do you think backward
   chaining needs to establish?
2. Backward chaining will also be asked to prove `needs_emergency_surgery`, which nothing in the
   rule base concludes. What should it return, and how much work should that cost?
3. Now suppose the hospital's full rule base has 104 rules instead of 24, because someone added
   the order sets for every other service. Which of the two engines does more work than before?

In [ ]:
def summarize(goal, rules, facts):
    forward_facts, _, forward_tests = forward_chain(rules, facts)
    forward_new = len(forward_facts) - len(facts)
    proved, chain, stats = backward_chain(rules, facts, goal)
    return forward_new, forward_tests, proved, stats["goals_visited"], len(chain)

# The rest of the hospital: order-set rules for services this patient is not on.
OTHER_SERVICE_RULES = []
for index in range(40):
    OTHER_SERVICE_RULES.append(rule(f"orderset-{index}-screen", [f"protocol_flag_{index}"],
                                    f"protocol_{index}_indicated"))
    OTHER_SERVICE_RULES.append(rule(f"orderset-{index}-order",
                                    [f"protocol_{index}_indicated", "age_above_75"],
                                    f"protocol_{index}_ordered"))

big_rule_base = SEPSIS_RULES + OTHER_SERVICE_RULES
big_facts = PATIENT_FACTS | {f"protocol_flag_{index}" for index in range(40)}

results = []
print(f"  {'rule base':>12}  {'goal':>26}  {'fwd new facts':>14}  {'fwd tests':>10}"
      f"  {'proved':>7}  {'bwd goals':>10}  {'bwd rules':>10}")
for label, rules, facts, goal in [
    ("24 rules", SEPSIS_RULES, PATIENT_FACTS, "transfer_to_icu"),
    ("24 rules", SEPSIS_RULES, PATIENT_FACTS, "needs_emergency_surgery"),
    ("104 rules", big_rule_base, big_facts, "transfer_to_icu"),
]:
    fwd_new, fwd_tests, proved, bwd_goals, bwd_rules = summarize(goal, rules, facts)
    results.append(bwd_rules)
    print(f"  {label:>12}  {goal:>26}  {fwd_new:>14}  {fwd_tests:>10}"
          f"  {str(proved):>7}  {bwd_goals:>10}  {bwd_rules:>10}")

if max(results) == 0:
    print()
    print("Every backward run above proved nothing and visited exactly one goal. That is the")
    print("placeholder talking, not a result: finish the TODO in the previous cell and rerun.")

This table is the whole reason both strategies exist. Going from 24 rules to 104, forward
chaining derives more than four times as many facts and tests four times as many rules, even
though the question never changed and none of the new rules can possibly bear on it. Backward
chaining does not move at all: it still visits 14 goals and uses 9 rules, because it only ever
looks at rules that conclude something it needs. And on `needs_emergency_surgery` — the "Failure
of Query" slide — it visits exactly one goal and returns False, while forward chaining derives
every fact in the hospital before it can tell you the same thing.

The trade is not free. Forward chaining answers *every* question at once, which is exactly what
you want for a surveillance engine watching 400 inpatients for deterioration. Backward chaining
answers one question cheaply, which is what you want for a consultation system sitting in front of
a clinician who asked something specific.

## Part 3 — Two rules, one patient, opposite conclusions

Here is a patient in shock: tachycardic, hypotensive, with distended neck veins and crackles at
both bases. Two rules in the fluid-management rule base match. One concludes that the patient is
volume depleted and should get a fluid bolus; the other concludes cardiogenic shock and says to
withhold fluids and start inotropes. Both rules are correct as written and both are satisfied.
Giving a two-litre bolus to a patient in cardiogenic pulmonary edema is a genuinely harmful
error, so which rule fires is not a stylistic question.

In [ ]:
FLUID_RULES = [
    rule("volume-depletion",
         ["tachycardia", "hypotension"],
         "give_fluid_bolus"),
    rule("cardiogenic-shock",
         ["tachycardia", "hypotension", "jugular_venous_distension", "pulmonary_crackles"],
         "withhold_fluids_start_inotropes"),
]

bedside_facts = {"tachycardia", "hypotension", "jugular_venous_distension", "pulmonary_crackles"}


def conflict_set(rules, facts):
    """Every rule whose antecedents are all satisfied: the agenda, before resolution."""
    return [r for r in rules if all(a in facts for a in r["antecedents"])]


agenda = conflict_set(FLUID_RULES, bedside_facts)
print("conflict set (both of these are satisfied right now):")
for r in agenda:
    print(f"  {r['name']:<20} {len(r['antecedents'])} antecedents -> {r['consequent']}")

print()
print("recommendation if the engine takes the first rule in list order:")
print(f"  {conflict_set(FLUID_RULES, bedside_facts)[0]['consequent']}")
print("recommendation if someone reorders the rule base:")
print(f"  {conflict_set(list(reversed(FLUID_RULES)), bedside_facts)[0]['consequent']}")

Editing the rule file changed the clinical recommendation. That is unacceptable, and it is why
production systems have an explicit conflict resolution step between recognize and act. The
"Conflict Resolution Strategies" slide lists several: order of appearance, recency of the matching
facts, explicit salience or priority, and specificity. Implement specificity — prefer the rule
with the most antecedents, on the grounds that a rule with more conditions describes a narrower
and better-characterized situation. The placeholder is "first in list order", which is itself a
real strategy and is the one that just gave us the wrong answer.

In [ ]:
def resolve_by_specificity(agenda):
    """Choose one rule from the conflict set. Prefer the most specific rule available."""
    if not agenda:
        return None

    # TODO: return the rule in `agenda` with the most antecedents, instead of simply the first
    #       one. `len(r["antecedents"])` is the specificity of rule r.
    return agenda[0]


print(f"  {'rule order':>22}  {'rule chosen':>20}  recommendation")
for label, rules in [("as written", FLUID_RULES), ("reversed", list(reversed(FLUID_RULES)))]:
    chosen = resolve_by_specificity(conflict_set(rules, bedside_facts))
    print(f"  {label:>22}  {chosen['name']:>20}  {chosen['consequent']}")

print()
print("Once specificity is in, both orderings give the same answer, and it is the safe one.")
print("Notice what specificity is really doing: it is a heuristic standing in for the fact that")
print("nobody wrote down that cardiogenic-shock should override volume-depletion.")

## Part 4 — `why()`: the ancestor of explainable AI

`forward_chain` recorded which rule produced each derived fact. That record is enough to
reconstruct the full derivation of anything the engine concluded, which is precisely the
"Explanation System" slide. Walk the chain from a fact back to its rule, then recursively back
through each of that rule's antecedents, and stop when you hit something the patient arrived
with. Every "explainable AI" claim you will hear about in Week 11 is trying to recover something
this rule engine got for free in 1975.

In [ ]:
def why(fact, chain, starting_facts, depth=0):
    """Print the derivation of `fact` by walking the recorded inference chain."""
    indent = "  " * depth
    if fact in starting_facts:
        print(f"{indent}{fact}  <- observed in the patient record")
        return
    if fact not in chain:
        print(f"{indent}{fact}  <- never derived")
        return

    producing_rule = chain[fact]
    print(f"{indent}{fact}  <- rule '{producing_rule['name']}' because:")
    for antecedent in producing_rule["antecedents"]:
        why(antecedent, chain, starting_facts, depth + 1)


print("Why does this patient need ICU transfer?")
why("transfer_to_icu", forward_inference_chain, PATIENT_FACTS)

print()
print("Why a nutrition consult?")
why("nutrition_consult", forward_inference_chain, PATIENT_FACTS)

Two things worth saying out loud about that output. First, the ICU explanation bottoms out
entirely in observed values — a temperature, a heart rate, a urine culture, a lactate, a mean
arterial pressure — with no black box anywhere in the tree, which is why regulators like rule
systems. Second, the explanation is only as good as the rule names, because `why()` reports the
rule that fired, not the reason the rule is correct. Nothing in the chain tells you *why* a MAP
below 65 matters.

In [ ]:
# TODO: pick any fact the engine derived and explain it. Try one from the sepsis path
#       (for example "give_broad_spectrum_antibiotics") and one from the incidental rules
#       (for example "social_work_referral"), and compare how deep the two derivations go.
fact_to_explain = "sepsis"

print(f"Why {fact_to_explain}?")
why(fact_to_explain, forward_inference_chain, PATIENT_FACTS)

print()
print("Facts available to explain:")
print("  " + ", ".join(sorted(forward_inference_chain)))

## Talk about it

1. Forward chaining fired `pressure_ulcer_risk`, `pharmacy_reconciliation`, and
   `social_work_referral` on a patient who was on their way to the ICU in septic shock. All three
   are true. If this engine were wired to your EHR's alert system, what have you just built, and
   what would the nurses do about it within a week?

2. Specificity picked the cardiogenic rule because it had four antecedents rather than two. Is
   "more conditions" actually a reason to trust a rule more? Construct a case where specificity
   picks the wrong rule, and say what strategy you would want instead.

3. `why()` produces a derivation, not a justification: it can tell you which rules fired but not
   whether the rules encode good medicine. Who is accountable when the chain is correct and the
   recommendation is wrong, and does the explanation help or hurt that conversation?

## Solutions

Completed versions of the three TODOs. These are markdown, not code cells, so scrolling here will
not overwrite your own work.

**Part 2 — the recursive case of `backward_chain`**

```python
    for r in rules:
        if r["consequent"] != goal:
            continue                      # This rule cannot possibly help; never look inside it.
        antecedents_proved = True
        for antecedent in r["antecedents"]:
            proved, chain, stats = backward_chain(
                rules, facts, antecedent, chain, stack | {goal}, stats)
            if not proved:
                antecedents_proved = False
                break                     # One failed antecedent kills the whole rule.
        if antecedents_proved:
            chain[goal] = r
            return True, chain, stats

    return False, chain, stats
```

**Part 3 — resolution by specificity**

```python
    # More antecedents means the rule describes a narrower, better-characterized situation.
    return max(agenda, key=lambda r: len(r["antecedents"]))
```

**Part 4 — there is no single right answer; this is one worth running**

```python
fact_to_explain = "give_broad_spectrum_antibiotics"
```

**Optional — a conflict resolution strategy with explicit priority**, which is what CLIPS calls
salience and what you will meet next week:

```python
def resolve_by_salience(agenda, salience):
    """salience maps a rule name to a number; higher fires first, ties broken by specificity."""
    return max(agenda, key=lambda r: (salience.get(r["name"], 0), len(r["antecedents"])))

print(resolve_by_salience(conflict_set(FLUID_RULES, bedside_facts),
                          {"cardiogenic-shock": 100})["consequent"])
```
